In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

PATH = r"D:\Weatherwalay\Day one\weather_hourly_clean.csv"
df = pd.read_csv(PATH, index_col="datetime", parse_dates=True)

HORIZON = 6

# ==================================================================
# FEATURES (identical to t+1 run)
# ==================================================================
feat = pd.DataFrame(index=df.index)
feat["y"] = df["temp_c"].shift(-HORIZON)

for lag in [1, 2, 3, 6, 12, 24]:
    feat[f"temp_lag_{lag}"] = df["temp_c"].shift(lag)

for col in ["dewpt_c", "hum", "pressure_mbar", "wind_kph"]:
    feat[f"{col}_lag_1"] = df[col].shift(1)

for w in [3, 6, 12, 24]:
    feat[f"temp_roll_mean_{w}"] = df["temp_c"].shift(1).rolling(w).mean()
    feat[f"temp_roll_std_{w}"]  = df["temp_c"].shift(1).rolling(w).std()
    feat[f"temp_roll_min_{w}"]  = df["temp_c"].shift(1).rolling(w).min()
    feat[f"temp_roll_max_{w}"]  = df["temp_c"].shift(1).rolling(w).max()

feat["temp_diff_1h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(2)
feat["temp_diff_3h"]  = df["temp_c"].shift(1) - df["temp_c"].shift(4)
feat["temp_diff_24h"] = df["temp_c"].shift(1) - df["temp_c"].shift(25)

feat["hour_sin"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
feat["hour_cos"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
feat["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
feat["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

feat["dewpt_spread"] = df["temp_c"].shift(1) - df["dewpt_c"].shift(1)

# Persistence baseline at t+6: predict temp(t+6) = temp(t-1)
feat["persistence"] = df["temp_c"].shift(1)

feat = feat.dropna()

# ==================================================================
# SAME SPLIT
# ==================================================================
split_train_val = "2016-07-01"
split_val_test  = "2016-10-01"

train = feat.loc[:split_train_val]
val   = feat.loc[split_train_val:split_val_test].iloc[1:]
test  = feat.loc[split_val_test:].iloc[1:]

X_tr, y_tr = train.drop(columns=["y", "persistence"]), train["y"]
X_vl, y_vl = val.drop(columns=["y", "persistence"]),   val["y"]
X_te, y_te = test.drop(columns=["y", "persistence"]),  test["y"]

# ==================================================================
# TRAIN
# ==================================================================
scaler = StandardScaler().fit(X_tr)
ridge = Ridge(alpha=1.0).fit(scaler.transform(X_tr), y_tr)

lgbm = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.02,
    num_leaves=15, max_depth=5, min_child_samples=50,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.7,
    reg_alpha=0.1, reg_lambda=0.5, random_state=42, verbose=-1,
)
lgbm.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], eval_metric="mae",
         callbacks=[lgb.early_stopping(50, verbose=False)])

# ==================================================================
# EVALUATE
# ==================================================================
def report(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name:<28} MAE: {mae:.3f}   RMSE: {rmse:.3f}")

print("=" * 60)
print(f"t+{HORIZON} TEMPERATURE FORECASTING")
print("=" * 60)

print("\nPersistence (temp(t+6) = temp(t-1)):")
report("  Test", y_te, test["persistence"])

print("\nRidge:")
report("  Train", y_tr, ridge.predict(scaler.transform(X_tr)))
report("  Val",   y_vl, ridge.predict(scaler.transform(X_vl)))
report("  Test",  y_te, ridge.predict(scaler.transform(X_te)))

print(f"\nLightGBM (best iter = {lgbm.best_iteration_}):")
report("  Train", y_tr, lgbm.predict(X_tr))
report("  Val",   y_vl, lgbm.predict(X_vl))
report("  Test",  y_te, lgbm.predict(X_te))

# ==================================================================
# SKILL vs PERSISTENCE
# ==================================================================
persist_mae = mean_absolute_error(y_te, test["persistence"])
ridge_mae   = mean_absolute_error(y_te, ridge.predict(scaler.transform(X_te)))
lgbm_mae    = mean_absolute_error(y_te, lgbm.predict(X_te))

print("\n=== Skill improvement over persistence ===")
print(f"Ridge:    {(persist_mae - ridge_mae) / persist_mae * 100:+.1f}%")
print(f"LightGBM: {(persist_mae - lgbm_mae) / persist_mae * 100:+.1f}%")

# ==================================================================
# SIDE-BY-SIDE WITH t+1 RESULTS
# ==================================================================
print("\n=== Comparison across horizons ===")
print(f"{'Horizon':<12} {'Persistence':<14} {'Ridge':<10} {'LightGBM':<10}")
print(f"{'t+1':<12} {0.717:<14.3f} {0.659:<10.3f} {0.713:<10.3f}")
print(f"{'t+6':<12} {persist_mae:<14.3f} {ridge_mae:<10.3f} {lgbm_mae:<10.3f}")